# 01 — Supervised fine-tuning (SFT) on CPU, end to end

This lab walks through the full SageMaker fine-tuning loop, from data in S3 to a model you can call, using CPU
instances only. It runs in any account without waiting for GPU capacity, and finishes in about 15 minutes.

**What you will do**

| Step | What happens | What it shows |
|---|---|---|
| 1. Data | generate 400 training / 30 test examples, write them as JSONL, upload them to S3 | how a training job receives your data (**input channels**) |
| 2. Train | one training job (`scripts/train.py`) scores the base model, trains a LoRA adapter on **Qwen3-0.6B** (20 steps), and scores the fine-tuned model on the same held-out examples | the anatomy of a job: container, script, data in, adapter + metrics out |
| 3. Capacity | the same job is launched 3 times in parallel: **on-demand**, **Spot**, and an instance-type **fallback** list | how to get compute when an instance type is scarce, and what Spot saves |
| 4. Results | compare base vs fine-tuned, in the notebook and in **MLflow** | what fine-tuning changed, and how to track runs |
| 5. Deploy | serve the adapter 3 ways: a **real-time** endpoint, a real-time endpoint with **instance pools** (fallback), and a **serverless** endpoint; then delete them | the deployment options and their trade-offs (always on vs cold start) |

**The task.** Turn a free-text restaurant booking request into strict JSON:

`"hey! Dana - four ppl friday 7pm if possible"` → `{"name": "Dana", "party_size": 4, "day": "Friday", "time": "19:00"}`

The base model gets the same instruction and usually gets the values roughly right, but not the exact format:
it wraps the JSON in a markdown code block, keeps "friday" lowercase, writes "7pm". The metric is **exact JSON
match**, so it starts near 0%; a few LoRA steps teach the format (expect ~100%).

**Where things end up** (all under your SageMaker default bucket, prefix `cpu-fine-tuning-techniques/`):

```
data/train/train.jsonl, data/test/test.jsonl     the dataset (step 1)
jobs/<job-name>/output/model/adapter/            the LoRA adapter (~9 MB): weights + config + tokenizer
jobs/<job-name>/output/model/metrics.json        base vs fine-tuned scores, example outputs
jobs/<job-name>/output/model/serving/            model.tar.gz for the endpoints (step 5)
```

**Before you start.** The account needs training quota for `ml.c7i.4xlarge` (on-demand, and Spot for the Spot
job) and endpoint quota for `ml.m5.xlarge` and serverless; `testing/check_quotas.py` lists them. Cost is
small: three ~4-minute CPU jobs plus endpoints for about 10 minutes, deleted by the last cell.

The model is small and the run short on purpose: this lab shows the mechanics. The same code pattern scales to
GPU instances and larger models; notebooks 02-04 apply it to other techniques (continued pre-training, DPO,
GRPO).

In [ ]:
%pip install -q "sagemaker==3.21.0"

In [ ]:
import json
import time

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
region = sess.boto_region_name
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]
bucket = sess.default_bucket()
prefix = "/".join(p for p in (sess.default_bucket_prefix, "cpu-fine-tuning-techniques") if p)
sm = boto3.client("sagemaker", region_name=region)
s3 = boto3.client("s3", region_name=region)
# MLflow: "" = reuse the account's MLflow App (or create one: several minutes, first time only);
# "arn:aws:sagemaker:..." = use that app; None = off. Jobs run the same either way.
from sm_helpers import mlflow_environment, mlflow_url, resolve_mlflow

MLFLOW_APP_ARN = ""
mlflow_arn = resolve_mlflow(sm, sess, MLFLOW_APP_ARN)
print(f"role:   {role}\nregion: {region}\noutput: s3://{bucket}/{prefix}/\nMLflow: {mlflow_arn or 'off'}")

## The data

Generate the examples, write them as JSONL (one `{"request": ..., "target": ...}` per line) and upload them to
S3. The training job gets them as two **input channels**, `train` and `test`: SageMaker copies each S3 prefix to
`/opt/ml/input/data/<channel>/` before the script starts, and `scripts/train.py` reads them from there. To use
your own data, upload files in the same format and point the channels at them.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "scripts")
from booking_data import INSTRUCTION, make_splits

train, test = make_splits(n_train=400, n_test=30)
print(f"instruction: {INSTRUCTION}\n{len(train)} training / {len(test)} test examples\n")
for ex in train[:3]:
    print(f"{ex['request']}\n  -> {ex['target']}")

data_uri = {}
for split, rows in (("train", train), ("test", test)):
    local = Path("data") / split / f"{split}.jsonl"
    local.parent.mkdir(parents=True, exist_ok=True)
    local.write_text("".join(json.dumps(r) + "\n" for r in rows))
    key = f"{prefix}/data/{split}/{local.name}"
    s3.upload_file(str(local), bucket, key)
    data_uri[split] = f"s3://{bucket}/{prefix}/data/{split}/"
    print(f"\n{split:5} -> {data_uri[split]}{local.name}", end="")

## Train and evaluate: three ways to get capacity

One training job does everything (`scripts/train.py`): it reads the two data channels, scores the base model,
trains the adapter, scores the fine-tuned model, and writes `adapter/` and `metrics.json` to `/opt/ml/model`,
which SageMaker uploads to S3 when the job ends. The same job is launched three times, in parallel, each asking
SageMaker for capacity in a different way:

| `capacity` | What SageMaker does | When to use it |
|---|---|---|
| `on-demand` | runs on exactly `INSTANCE_TYPE` | the default; predictable |
| `spot` | runs on `INSTANCE_TYPE` using spare (Spot) capacity, waiting up to `max_wait` for it; can be interrupted | cheaper; for jobs that can wait, or resume from checkpoints |
| `fallback` | runs on the first type in `FALLBACK_TYPES` that has capacity (`instance_preferences`) | when one type is often unavailable (typical for GPUs) |

SageMaker rejects a fallback list if any type in it has no training quota, so types without quota are dropped
first. Outputs are stored uncompressed so the endpoints below can use them directly.

In [ ]:
from botocore.config import Config
from sagemaker.core.shapes import InstancePreference
from sagemaker.core.training.configs import Compute, InputData, OutputDataConfig, SourceCode, StoppingCondition
from sagemaker.train import ModelTrainer

STEPS = 20
INSTANCE_TYPE = "ml.c7i.4xlarge"
FALLBACK_TYPES = ["ml.c7i.4xlarge", "ml.m7i.4xlarge", "ml.c6i.4xlarge", "ml.m6i.4xlarge", "ml.m5.4xlarge"]

try:
    # adaptive retries: notebooks started together can hit the Service Quotas rate limit
    sq = boto3.client("service-quotas", region_name=region, config=Config(retries={"max_attempts": 10, "mode": "adaptive"}))
    quotas = {q["QuotaName"]: q["Value"]
              for page in sq.get_paginator("list_service_quotas").paginate(ServiceCode="sagemaker") for q in page["Quotas"]}
    FALLBACK_TYPES = [t for t in FALLBACK_TYPES if quotas.get(f"{t} for training job usage", 0) >= 1] or FALLBACK_TYPES
except Exception as e:  # role cannot read Service Quotas: send the full list
    print(f"quota check skipped: {e}")


def make_trainer(capacity):
    if capacity == "fallback":
        compute = Compute(instance_preferences=[InstancePreference(instance_type=t) for t in FALLBACK_TYPES],
                          instance_count=1, volume_size_in_gb=30)
    else:
        compute = Compute(instance_type=INSTANCE_TYPE, instance_count=1, volume_size_in_gb=30,
                          enable_managed_spot_training=capacity == "spot")
    return ModelTrainer(
        training_image=f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-training:2.8-cpu-py312",
        source_code=SourceCode(source_dir="scripts", requirements="requirements.txt", entry_script="train.py"),
        hyperparameters={"steps": STEPS, "model_id": "Qwen/Qwen3-0.6B"},
        environment=mlflow_environment(mlflow_arn),  # the job logs to MLflow when an app is set
        compute=compute,
        # Spot only: max_wait is the total time allowed, including waiting for Spot capacity; past it the job
        # stops instead of waiting for capacity indefinitely
        stopping_condition=StoppingCondition(max_runtime_in_seconds=1200,
                                             max_wait_time_in_seconds=1800 if capacity == "spot" else None),
        output_data_config=OutputDataConfig(s3_output_path=f"s3://{bucket}/{prefix}/jobs", compression_type="NONE"),
        base_job_name=f"cpu-ft-sft-{capacity}",
        role=role,
        sagemaker_session=sess,
    )


jobs = {}
for capacity in ("on-demand", "spot", "fallback"):
    trainer = make_trainer(capacity)
    # The two S3 prefixes from "The data" become /opt/ml/input/data/train and /opt/ml/input/data/test in the job
    trainer.train(input_data_config=[InputData(channel_name=split, data_source=uri) for split, uri in data_uri.items()],
                  wait=False)
    jobs[capacity] = trainer._latest_training_job.training_job_name
    print(f"{capacity:10} submitted: {jobs[capacity]}")
print(f"fallback list: {FALLBACK_TYPES}")

### Wait for the three jobs

The jobs run in SageMaker independently of this notebook. If the kernel restarts, re-run the setup cell and this
one: it finds the latest job of each kind.

In [ ]:
jobs = globals().get("jobs") or {
    c: sm.list_training_jobs(NameContains=f"cpu-ft-sft-{c}", SortBy="CreationTime", SortOrder="Descending",
                             MaxResults=1)["TrainingJobSummaries"][0]["TrainingJobName"]
    for c in ("on-demand", "spot", "fallback")}
GRACE_S = 600  # once on-demand is done, wait at most this long for the others (e.g. Spot without capacity)
last, done_at = {}, None
while True:
    desc = {c: sm.describe_training_job(TrainingJobName=n) for c, n in jobs.items()}
    for c, d in desc.items():
        line = f"{d['TrainingJobStatus']}/{d['SecondaryStatus']}"
        if last.get(c) != line:
            print(time.strftime("%H:%M:%S"), f"{c:10}", line, flush=True)
            last[c] = line
    finished = {c for c, d in desc.items() if d["TrainingJobStatus"] in ("Completed", "Failed", "Stopped")}
    if len(finished) == len(desc):
        break
    if "on-demand" in finished:
        done_at = done_at or time.time()
        if time.time() - done_at > GRACE_S:
            print(f"\nstill running: {sorted(set(desc) - finished)}. Continuing with the on-demand model; "
                  "re-run this cell later to see how they ended.")
            break
    time.sleep(15)


def phases(d):
    out = {}
    for t in d.get("SecondaryStatusTransitions", []):
        end = t.get("EndTime") or d.get("TrainingEndTime")
        if end:
            out[t["Status"]] = (end - t["StartTime"]).total_seconds()
    return out


print(f"\n{'capacity':10} {'status':10} {'instance':16} {'to start':>8} {'training':>8} {'billed':>7} {'spot savings':>12}")
for c, d in desc.items():
    rc, p = d["ResourceConfig"], phases(d)
    savings = ""
    if c == "spot" and d.get("TrainingTimeInSeconds"):
        savings = f"{1 - d['BillableTimeInSeconds'] / d['TrainingTimeInSeconds']:.0%}"
    print(f"{c:10} {d['TrainingJobStatus']:10} {rc.get('SelectedInstanceType') or rc.get('InstanceType'):16} "
          f"{p.get('Starting', 0) + p.get('Downloading', 0):7.0f}s {p.get('Training', 0):7.0f}s "
          f"{d.get('BillableTimeInSeconds', 0):6}s {savings:>12}")

**How to read the table.** One row per capacity option:

| Column | Meaning |
|---|---|
| status | `Completed`, or `Failed`/`Stopped` (e.g. Spot found no capacity within `max_wait`) |
| instance | the instance the job ran on; for `fallback`, the type SageMaker picked from the list |
| to start | time from submission until the script started: waiting for capacity, starting the instance, downloading the container and data |
| training | time the script ran (install requirements, download the model, evaluate, train, evaluate) |
| billed | seconds you pay for. On-demand and fallback: about the job's run time. Spot: the same run time at the Spot discount, shown as "spot savings" (`1 - billable time / training time`) |

The three jobs train the same model on the same data, so their results match; only the capacity path differs.
Most of the time is startup, not training: that is the fixed cost of a training job, and why short experiments
are often run in a notebook or on a dev instance first.

## Results

In [ ]:
def metrics_of(job_name):
    d = sm.describe_training_job(TrainingJobName=job_name)
    prefix_uri = d["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/") + "/"
    b, k = prefix_uri[5:].split("/", 1)
    return prefix_uri, json.loads(s3.get_object(Bucket=b, Key=k + "metrics.json")["Body"].read())


model_prefix, metrics = metrics_of(jobs["on-demand"])
print(f"adapter: {model_prefix}adapter/\n"
      f"         https://s3.console.aws.amazon.com/s3/buckets/{model_prefix[5:].split('/', 1)[0]}"
      f"?prefix={model_prefix[5:].split('/', 1)[1]}adapter/\n")
print(f"{'':10} {'exact':>6} " + " ".join(f"{f:>10}" for f in ("name", "party_size", "day", "time")))
for which in ("base", "finetuned"):
    m = metrics[which]
    print(f"{which:10} {m['exact_match']:6.0%} " + " ".join(f"{m['acc_' + f]:10.0%}" for f in ("name", "party_size", "day", "time")))
print(f"\n{metrics['steps']} steps, {metrics['seconds_training']} s training\n")
for ex in metrics["examples"][:2]:
    print(f"request:   {ex['request']}\nbase:      {ex['base']!r}\nfinetuned: {ex['finetuned']}\n")
for c in ("spot", "fallback"):
    if desc[c]["TrainingJobStatus"] == "Completed":
        print(f"{c:10} fine-tuned exact match: {metrics_of(jobs[c])[1]['finetuned']['exact_match']:.0%}")

### How to read the results

| Row | Model |
|---|---|
| `base` | Qwen3-0.6B as downloaded, given the same instruction |
| `finetuned` | the same model + the LoRA adapter trained in this job |

| Column | What it measures (on the 30 test requests, never used in training) |
|---|---|
| exact | the answer parses as JSON and equals the target exactly: bare JSON (no markdown code block), `"Friday"` capitalized, 24-hour `"19:00"` |
| name, party_size, day, time | per field: the share of answers where that field is right. The base model often has the values right inside a code block or in another format, which counts as wrong here |

The example requests below show the difference: the base model's answer is close but not usable by a program
expecting strict JSON; the fine-tuned answer is. Only the adapter changed: ~2.3 M trained parameters (~0.4% of the
model), saved as a ~9 MB file.

### In MLflow

Each job also logs its parameters, the training loss per step, and the base and fine-tuned scores, one run per
job in the experiment `cpu-fine-tuning-techniques` (notebooks 02-04 log there too). The link is valid for a few
minutes; re-run the cell for a new one.

In [ ]:
# Compare the three runs in MLflow: experiment "cpu-fine-tuning-techniques", one run per job
print(mlflow_url(sm, mlflow_arn) or "MLflow is off")

## Serve it: three ways to deploy

All three endpoints run the same model (base model from the Hugging Face Hub + the adapter from the on-demand job,
merged at startup by `serving/inference.py`) and differ only in how capacity is provided:

| endpoint | What SageMaker does | Billing |
|---|---|---|
| `realtime` | one `ml.m5.xlarge` instance, always on | per instance-hour |
| `realtime-pools` | the first type in a priority list that has capacity (`InstancePools`, up to 5) | per instance-hour |
| `serverless` | starts compute on demand; the first request after idle time waits for a cold start | per request (compute time) |

They take a few minutes to create. The real-time ones bill per hour until deleted (last cell).

In [ ]:
import tarfile
import tempfile
from pathlib import Path

# Package the adapter (~9 MB) + the handler code as model.tar.gz: serverless endpoints need a tarball
# (ModelDataUrl); the base model is downloaded from the Hub when the container starts.
mb, mk = model_prefix[5:].split("/", 1)
work = Path(tempfile.mkdtemp())
for obj in s3.list_objects_v2(Bucket=mb, Prefix=mk + "adapter/")["Contents"]:
    dest = work / "adapter" / obj["Key"].rsplit("/", 1)[-1]
    dest.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(mb, obj["Key"], str(dest))
with tarfile.open(work / "model.tar.gz", "w:gz") as tar:
    tar.add(work / "adapter", arcname="adapter")
    for name in ("inference.py", "requirements.txt"):
        tar.add(f"serving/{name}", arcname=f"code/{name}")
s3.upload_file(str(work / "model.tar.gz"), mb, mk + "serving/model.tar.gz")

container = {
    "Image": f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-inference:2.6-cpu-py312",
    "ModelDataUrl": f"s3://{mb}/{mk}serving/model.tar.gz",
    "Environment": {"SAGEMAKER_PROGRAM": "inference.py", "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
                    "SAGEMAKER_MODEL_SERVER_WORKERS": "1"},
}
variants = {
    "realtime": {"InstanceType": "ml.m5.xlarge", "InitialInstanceCount": 1},
    "realtime-pools": {"InstancePools": [{"InstanceType": t, "Priority": i + 1}
                                         for i, t in enumerate(["ml.m5.xlarge", "ml.m7i.xlarge", "ml.c7i.2xlarge"])],
                       "InitialInstanceCount": 1, "VariantInstanceProvisionTimeoutInSeconds": 1800},
    "serverless": {"ServerlessConfig": {"MemorySizeInMB": 6144, "MaxConcurrency": 1}},
}
stamp = time.strftime("%Y%m%d-%H%M%S")
endpoints = {}
for kind, variant in variants.items():
    name = f"cpu-ft-sft-{kind}-{stamp}"
    sm.create_model(ModelName=name, ExecutionRoleArn=role, PrimaryContainer=container)
    sm.create_endpoint_config(EndpointConfigName=name,
                              ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": name, **variant}])
    sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
    endpoints[kind] = name
t0 = time.time()
for kind, name in endpoints.items():
    try:
        sm.get_waiter("endpoint_in_service").wait(EndpointName=name, WaiterConfig={"Delay": 20, "MaxAttempts": 90})
        print(f"{kind:15} in service after {time.time() - t0:.0f}s")
    except Exception:
        print(f"{kind:15} {sm.describe_endpoint(EndpointName=name).get('FailureReason')}")

In [ ]:
from botocore.config import Config

runtime = boto3.client("sagemaker-runtime", region_name=region,
                       config=Config(read_timeout=120, retries={"max_attempts": 2}))
request = "hey! Dana - four ppl friday 7pm if possible"
for kind, name in endpoints.items():
    for attempt in ("first", "second"):
        t = time.time()
        try:
            resp = runtime.invoke_endpoint(EndpointName=name, ContentType="application/json",
                                           Body=json.dumps({"request": request}))
            out = json.loads(resp["Body"].read())["booking"]
        except Exception as e:
            out = f"error: {str(e)[:120]}"
        print(f"{kind:15} {attempt:6} {time.time() - t:5.1f}s  {out}")

**How to read the output.** Each endpoint is called twice with the same request; the answer should be the same
JSON from all three. The time is the full round trip from this notebook:

* `realtime` and `realtime-pools`: a few seconds both times. The instance is already running with the model
  loaded, and you pay for it per hour whether it is used or not.
* `serverless`: the first call includes a **cold start** (SageMaker starts compute, downloads the container and
  the base model, loads the adapter: tens of seconds); the second call is warm. You pay only while requests are
  processed, so it suits low or irregular traffic that can tolerate an occasional slow first request.

`realtime-pools` behaves like `realtime` here; its value shows when the first instance type has no capacity and
SageMaker moves down the list.

In [ ]:
# Clean up the endpoints (the real-time ones bill per hour while they exist)
for name in endpoints.values():
    sm.delete_endpoint(EndpointName=name)
    sm.delete_endpoint_config(EndpointConfigName=name)
    sm.delete_model(ModelName=name)
    print("deleted", name)

## What you saw

* **Data** goes to the job through S3 input channels; **results** (adapter, metrics) come back to S3 from
  `/opt/ml/model`.
* One **training job** can evaluate, train and evaluate again, so every run carries its own before/after numbers,
  also in **MLflow**.
* **Capacity** is a choice: on-demand for predictability, Spot for cost, a fallback list when a type is scarce.
* A LoRA adapter is a small file on top of a public base model; **deploying** it means base model + adapter, on a
  real-time endpoint (always on) or a serverless one (pay per request, cold starts).

Next: `02-continued-pretraining.ipynb`, `03-dpo.ipynb` and `04-grpo.ipynb` reuse this pattern for other techniques.